In [ ]:
import os
import json
import math
import platform
from datetime import datetime

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

N_BISTABLE_POOL_TARGET = 1000
MAX_TRIES = 400000
PROGRESS_EVERY_TRIES = 5000
PROGRESS_EVERY_ACCEPTED = 50

SOC_TOL = 1e-6
SOC_RHO_MIN = 1e-6
SOC_GRID_N = 4001
SOC_REFINE_N = 3

PSI_GRID_N_LIST = [101, 201, 401]
BARPSI_BISECT_ITERS_LIST = [30, 60, 90]
PSI0_BUFFER_LIST = [0.01, 0.03, 0.05]
N_USED_LIST = [100, 300, 500, 1000]
N_USED_D3_LIST = [100, 300, 500, 1000]
SAMPLE_DESIGN = "conditional_bistable_threshold_location"
THRESHOLD_LOCATION_U_RANGE = (0.1, 0.9)

assert max(N_USED_LIST) <= N_BISTABLE_POOL_TARGET
assert max(N_USED_D3_LIST) <= N_BISTABLE_POOL_TARGET
assert SAMPLE_DESIGN == "conditional_bistable_threshold_location"

T = 60.0
DT = 0.05
BISTABLE_MARGIN = 1e-3
ESCAPE_TOL = 0.02
CONFIRM_WINDOW = 10.0
D2_RATIO_DENOM_MIN = 1e-8

D_GAMMA_LIST = [0.005, 0.01, 0.02]
REFORM_SIZE_LIST = [0.01, 0.03, 0.05, 0.07, 0.10]
START_TIME_LIST = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]
REFORM_FREQ_LIST = [10.0, 20.0, 40.0]
MAX_REFORMS_LIST = [5, 10]
SEED = 20260312

PSI_HI = 0.999
EPS_PSI_L = 0.02
EPS_PSI_HI = 1e-4

MC_NAME = "mc_block3_intervention_hysteresis"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")
BASE_OUT_DIR = os.path.join(os.getcwd(), "results")
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
FIGS_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)

MC_CONFIG = {
    "N_COUNTRIES": N_BISTABLE_POOL_TARGET,
    "psi_grid_n": 401,
    "rho_bar_range": (1.0, 5.0),
    "a_range": (0.5, 2.0),
    "c_lambda_range": (0.1, 1.5),
    "eta_lambda_range": (1.1, 3.0),
    "c_ell_range": (0.05, 0.5),
    "eta_ell_range": (1.1, 3.0),
    "psi_L_range": (0.0, 0.4),
    "Gamma_range": (0.01, 0.08),
    "k_range": (0.1, 0.5),
    "s_N_range": (0.05, 0.5),
    "kappa_c_range": (0.2, 2.0),
    "sigma_range": (0.03, 0.12),
    "omega_mult_range": (0.5, 2.0),
    "max_bad_share": 0.05,
    "eps_solv": 1e-12,
    "tol_mono": 1e-10,
    "tol_kkt": 1e-10,
    "tol_g": 1e-12,
    "DERIV_RHO_MIN": 1e-8,
    "OMEGA_MIN": 1e-8,
}

SIM_CONFIG = {
    "dt": DT,
    "eps_psi": 1e-4,
    "eps_g": 1e-4,
    "stable_window": 200,
}

IC_NEAR_EPS = 0.05
IC_FAR_BELOW_MULT = 0.50
IC_FAR_ABOVE_MULT = 1.00
IC_FAR_ABOVE_CAP_MULT = 0.50

def g_scale(gstar):
    return 1.0 + abs(gstar)

PLOTS = []

def save_df(df: pd.DataFrame, tag: str) -> str:
    path = os.path.join(OUT_DIR, f"{MC_NAME}_{tag}_{RUN_TS}.csv")
    df.to_csv(path, index=False)
    return path

def save_fig(fig, name: str) -> str:
    png = os.path.join(FIGS_DIR, f"{name}.png")
    pdf = os.path.join(FIGS_DIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    PLOTS.append({"name": name, "png": png, "pdf": pdf})
    return pdf

def mono_nondec(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) >= -tol))

def mono_noninc(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) <= tol))

def r_func(rho, a, b):
    return a * rho - b * rho**2

def r_prime(rho, a, b):
    return a - 2.0 * b * rho

def r_second(b):
    return -2.0 * b

def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)

def lam_prime(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= MC_CONFIG["DERIV_RHO_MIN"]
    if np.any(valid):
        v = arr[valid]
        out[valid] = np.exp(-c_lam * v**eta_lam) * (c_lam * eta_lam * v**(eta_lam - 1.0))
    return out if arr.ndim > 0 else float(out)

def lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        f = np.exp(-c_lam * v**eta_lam)
        g = c_lam * eta_lam * v**(eta_lam - 1.0)
        gp = c_lam * eta_lam * (eta_lam - 1.0) * v**(eta_lam - 2.0)
        out[valid] = f * (gp - g**2)
    return out if arr.ndim > 0 else float(out)

def ell_func(rho, c_ell, eta_ell):
    return c_ell * rho**eta_ell

def ell_prime(rho, c_ell, eta_ell):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= MC_CONFIG["DERIV_RHO_MIN"]
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * v**(eta_ell - 1.0)
    return out if arr.ndim > 0 else float(out)

def ell_second(rho, c_ell, eta_ell, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * (eta_ell - 1.0) * v**(eta_ell - 2.0)
    return out if arr.ndim > 0 else float(out)

def B_func(rho, c_lam, eta_lam, c_ell, eta_ell):
    return lam_prime(rho, c_lam, eta_lam) * ell_func(rho, c_ell, eta_ell) + lam_func(rho, c_lam, eta_lam) * ell_prime(rho, c_ell, eta_ell)

def B_prime(rho, c_lam, eta_lam, c_ell, eta_ell, DERIV_RHO_MIN):
    lam = lam_func(rho, c_lam, eta_lam)
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam_pp = lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN)
    ell = ell_func(rho, c_ell, eta_ell)
    ell_p = ell_prime(rho, c_ell, eta_ell)
    ell_pp = ell_second(rho, c_ell, eta_ell, DERIV_RHO_MIN)
    return lam_pp * ell + 2.0 * lam_p * ell_p + lam * ell_pp

def global_soc_margin(theta, cfg):
    """Dense-grid global SOC screen with local refinement at the psi=0 benchmark."""
    def margin(rho):
        bp = np.asarray(B_prime(
            rho, theta["c_lambda"], theta["eta_lambda"],
            theta["c_ell"], theta["eta_ell"], cfg["DERIV_RHO_MIN"]
        ), dtype=float)
        return 2.0 * theta["b"] - np.maximum(0.0, -bp)

    grid = np.linspace(SOC_RHO_MIN, theta["rho_bar"], SOC_GRID_N)
    values = np.asarray(margin(grid), dtype=float)
    if not np.all(np.isfinite(values)):
        return -np.inf

    minimum = float(np.min(values))
    step = grid[1] - grid[0]
    for idx in np.argsort(values)[:SOC_REFINE_N]:
        lower = max(SOC_RHO_MIN, grid[idx] - step)
        upper = min(theta["rho_bar"], grid[idx] + step)
        result = minimize_scalar(
            lambda x: float(margin(x)),
            bounds=(lower, upper),
            method="bounded",
            options={"xatol": 1e-12},
        )
        if result.success and np.isfinite(result.fun):
            minimum = min(minimum, float(result.fun))
    return minimum

def solve_rho_star(psi_val, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell, tol=1e-12, max_iter=300):
    def F(rho):
        return r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)

    F0 = r_prime(0.0, a, b) - (1.0 - psi_val) * B_func(0.0, c_lam, eta_lam, c_ell, eta_ell)
    Fhi = r_prime(rho_bar, a, b) - (1.0 - psi_val) * B_func(rho_bar, c_lam, eta_lam, c_ell, eta_ell)

    if not np.isfinite(Fhi):
        return np.nan, "nonfinite"

    if abs(F0) <= tol:
        return 0.0, "corner_low"
    if abs(Fhi) <= tol:
        return rho_bar, "corner_high"

    if F0 * Fhi < 0.0:
        lo, hi = 0.0, rho_bar
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            f_mid = F(mid)
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (hi - lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(F0):
                lo = mid
            else:
                hi = mid
        return np.nan, "no_converge"

    if F0 > 0.0 and Fhi > 0.0:
        return rho_bar, "corner_high"
    if F0 < 0.0 and Fhi < 0.0:
        return 0.0, "corner_low"

    return np.nan, "no_bracket"

def check_kkt_status(psi_val, rho, status, a, b, c_lam, eta_lam, c_ell, eta_ell, tol_kkt=1e-10):
    F_val = r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)
    if status == "interior":
        return abs(F_val) <= tol_kkt
    elif status == "corner_low":
        return F_val <= tol_kkt
    elif status == "corner_high":
        return F_val >= -tol_kkt
    return False

def draw_primitives(rng_local, cfg):
    return {
        "rho_bar": rng_local.uniform(*cfg["rho_bar_range"]),
        "a": rng_local.uniform(*cfg["a_range"]),
        "c_lambda": rng_local.uniform(*cfg["c_lambda_range"]),
        "eta_lambda": rng_local.uniform(*cfg["eta_lambda_range"]),
        "c_ell": rng_local.uniform(*cfg["c_ell_range"]),
        "eta_ell": rng_local.uniform(*cfg["eta_ell_range"]),
        "psi_L": rng_local.uniform(*cfg["psi_L_range"]),
        "Gamma": rng_local.uniform(*cfg["Gamma_range"]),
        "k": rng_local.uniform(*cfg["k_range"]),
        "s_N": rng_local.uniform(*cfg["s_N_range"]),
        "kappa_c": rng_local.uniform(*cfg["kappa_c_range"]),
    }

def draw_admissible_primitives(rng_local, cfg, max_attempts=10000):
    for attempt in range(max_attempts):
        theta = draw_primitives(rng_local, cfg)
        rho_bar = theta["rho_bar"]
        c_ell = theta["c_ell"]
        eta_ell = theta["eta_ell"]
        k = theta["k"]

        ell_bar = ell_func(rho_bar, c_ell, eta_ell)
        if np.isfinite(ell_bar) and (0.0 < ell_bar < 1.0) and (k * ell_bar < 1.0):
            theta["b"] = theta["a"] / (2.0 * rho_bar)
            return theta
    raise RuntimeError("Failed to generate admissible primitives.")

def draw_dyn_params(rng_local, cfg):
    return {
        "sigma": rng_local.uniform(*cfg["sigma_range"]),
        "omega_mult": rng_local.uniform(*cfg["omega_mult_range"]),
        "mu_implied": np.nan,
    }

def g_star_stationary(psi, Gamma_eff, kappa_c):
    if (not np.isfinite(psi)) or (not np.isfinite(Gamma_eff)) or (not np.isfinite(kappa_c)):
        return np.nan
    if Gamma_eff <= 0.0:
        return 0.0
    if psi <= 0.0:
        return float(Gamma_eff)
    disc = 1.0 + 4.0 * kappa_c * psi * Gamma_eff
    if disc <= 0.0:
        return 0.0
    return float((-1.0 + np.sqrt(disc)) / (2.0 * kappa_c * psi))

def build_manifold_objects(theta, cfg, include_structural_drag=True):
    DERIV_RHO_MIN = cfg["DERIV_RHO_MIN"]

    rho_bar, a, b = theta["rho_bar"], theta["a"], theta["b"]
    c_lam, eta_lam = theta["c_lambda"], theta["eta_lambda"]
    c_ell, eta_ell = theta["c_ell"], theta["eta_ell"]
    psi_L, s_N, Gamma, k, kappa_c = theta["psi_L"], theta["s_N"], theta["Gamma"], theta["k"], theta["kappa_c"]

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])

    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell)
    base_ok = st0 in {"interior", "corner_low", "corner_high"}
    if not base_ok or (not np.isfinite(rho0)):
        n = psi_grid.size
        return {
            "psi_grid": psi_grid,
            "solver_ok_endpoints": False,
            "excluded_bad_share": True,
            "bad_share": 1.0,
            "valid_mask": np.zeros(n, dtype=bool),
            "rho_star": np.full(n, np.nan),
            "rK": np.full(n, np.nan),
            "Gamma_eff": np.full(n, np.nan),
            "g_star": np.full(n, np.nan),
            "W_star": np.full(n, np.nan),
            "ell": np.full(n, np.nan),
            "lam": np.full(n, np.nan),
            "solv_ok": False,
            "gstar_ok": False,
            "mono_ok": False,
            "deriv_ok": False,
            "omega_base": np.nan,
            "solver_status": np.array(["base_fail"] * n, dtype=object),
        }

    base_drag = k * lam_func(rho0, c_lam, eta_lam) * ell_func(rho0, c_ell, eta_ell)

    n = psi_grid.size
    rho_star = np.full(n, np.nan)
    solver_status = np.empty(n, dtype=object)
    solver_ok = np.zeros(n, dtype=bool)

    deriv_ok_flag = True
    for i, psi in enumerate(psi_grid):
        rho_i, status = solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell)
        solver_status[i] = status
        solver_ok[i] = status in {"interior", "corner_low", "corner_high"}
        rho_star[i] = rho_i if solver_ok[i] else np.nan

        if status == "interior":
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                deriv_ok_flag = False
            else:
                Bp = B_prime(rho_i, c_lam, eta_lam, c_ell, eta_ell, DERIV_RHO_MIN)
                if not np.isfinite(Bp):
                    deriv_ok_flag = False

    solver_ok_endpoints = bool(solver_ok[0] and solver_ok[-1])

    lam = lam_func(rho_star, c_lam, eta_lam)
    ell = ell_func(rho_star, c_ell, eta_ell)
    drag = k * lam * ell - base_drag
    Gamma_eff = Gamma - drag if include_structural_drag else np.full(n, Gamma, dtype=float)
    rK = r_func(rho_star, a, b) - (1.0 - psi_grid) * lam * ell
    g_star = np.array([g_star_stationary(psi_grid[i], Gamma_eff[i], kappa_c) for i in range(n)])
    W_star = rK - g_star

    valid = solver_ok.copy()
    valid &= np.isfinite(lam) & np.isfinite(ell)
    valid &= np.isfinite(rK) & np.isfinite(Gamma_eff)
    valid &= np.isfinite(g_star) & np.isfinite(W_star)

    bad_share = 1.0 - float(np.mean(valid))
    excluded_bad_share = bool(bad_share > cfg["max_bad_share"])

    solv_grid = (s_N - psi_grid * k * ell) > (cfg["eps_solv"] * max(1.0, s_N))
    g_star_ok = np.isfinite(g_star) & (g_star >= -cfg["tol_g"])

    if np.any(valid) and solver_ok_endpoints and (not excluded_bad_share):
        solv_ok = bool(np.all(solv_grid[valid]))
        gstar_ok = bool(np.all(g_star_ok[valid]))
        mono_ok = bool(
            mono_nondec(rho_star[valid], tol=1e-7)
            and mono_nondec(drag[valid], tol=1e-9)
            and mono_noninc(g_star[valid], tol=1e-9)
            and mono_nondec(W_star[valid], tol=1e-9)
        )
    else:
        solv_ok = False
        gstar_ok = False
        mono_ok = False

    Wabs = np.abs(W_star[valid]) if np.any(valid) else np.array([])
    omega_base = float(np.median(Wabs)) if Wabs.size else np.nan
    if (not np.isfinite(omega_base)) or (omega_base <= 0.0):
        omega_base = np.nan
    else:
        omega_base = max(omega_base, cfg["OMEGA_MIN"])

    return {
        "psi_grid": psi_grid,
        "solver_ok_endpoints": solver_ok_endpoints,
        "excluded_bad_share": excluded_bad_share,
        "bad_share": float(bad_share),
        "valid_mask": valid,
        "rho_star": rho_star,
        "rK": rK,
        "Gamma_eff": Gamma_eff,
        "g_star": g_star,
        "W_star": W_star,
        "ell": ell,
        "lam": lam,
        "solv_ok": bool(solv_ok),
        "gstar_ok": bool(gstar_ok),
        "mono_ok": bool(mono_ok),
        "deriv_ok": bool(deriv_ok_flag),
        "omega_base": omega_base,
        "solver_status": solver_status,
    }

def make_grid_lookups(objs, theta):
    psi_grid = objs["psi_grid"]
    valid = objs["valid_mask"]

    if not (valid[0] and valid[-1]):
        raise ValueError("Grid coverage fail: Endpoints are not valid.")

    x = psi_grid[valid]
    rK_arr = objs["rK"][valid]
    Ge_arr = objs["Gamma_eff"][valid]
    gstar_arr = objs["g_star"][valid]
    W_arr = objs["W_star"][valid]

    def rK_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, rK_arr))

    def Ge_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, Ge_arr))

    def gstar_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, gstar_arr))

    def W_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, W_arr))

    return rK_of_psi, Ge_of_psi, gstar_of_psi, W_of_psi

def classify_regime(W_adjusted_floor, W_adjusted_top, margin):
    """
    Classifies the structural configuration under the canonical terminology.
    """
    if not (np.isfinite(W_adjusted_floor) and np.isfinite(W_adjusted_top)):
        return "unclassified"
    if W_adjusted_floor >= -margin:
        return "born_high"
    if W_adjusted_top <= margin:
        return "born_low"
    if W_adjusted_floor <= -margin and W_adjusted_top >= margin:
        return "bistable_interior"
    return "unclassified"

def find_barpsi(psi_grid, W_grid, psi_L, mu, bisect_iters, valid_mask=None):
    """
    Bisection locator of barpsi solving W*(psi) = mu.
    """
    psi_arr = np.asarray(psi_grid, dtype=float)
    W_arr = np.asarray(W_grid, dtype=float)
    if valid_mask is None:
        valid = np.isfinite(psi_arr) & np.isfinite(W_arr)
    else:
        valid = np.asarray(valid_mask, dtype=bool) & np.isfinite(psi_arr) & np.isfinite(W_arr)

    if np.sum(valid) < 2:
        return np.nan, "unclassified", np.nan, np.nan

    x = psi_arr[valid]
    W = W_arr[valid]
    if (x[0] > psi_L + 1e-12) or (x[-1] < 1.0 - 1e-12):
        return np.nan, "unclassified", np.nan, np.nan

    W_floor = float(np.interp(psi_L, x, W))
    W_top = float(np.interp(1.0, x, W))

    if not (np.isfinite(W_floor) and np.isfinite(W_top)):
        return np.nan, "unclassified", W_floor, W_top

    dW = np.diff(W)
    if dW.size and np.min(dW) < -1e-8:
        return np.nan, "unclassified_nonmonotone", W_floor, W_top

    W_adjusted_floor = W_floor - mu
    W_adjusted_top = W_top - mu

    regime = classify_regime(W_adjusted_floor, W_adjusted_top, BISTABLE_MARGIN)
    if regime != "bistable_interior":
        return np.nan, regime, W_floor, W_top

    lo, hi = float(psi_L), 1.0
    for _ in range(int(bisect_iters)):
        mid = 0.5 * (lo + hi)
        W_mid = float(np.interp(mid, x, W)) - mu
        if W_mid >= 0.0:
            hi = mid
        else:
            lo = mid
    return float(hi), "bistable_interior", W_floor, W_top

def simulate_psi_path(psi0, psi_L, W_of_psi, sigma, omega, mu, T, dt, reforms=None):
    n_steps = int(np.ceil(T / dt))
    t = np.linspace(0.0, n_steps * dt, n_steps + 1)
    psi = np.empty(n_steps + 1, dtype=float)
    psi[0] = float(np.clip(psi0, psi_L, 1.0))

    reforms = reforms or []
    reforms_sorted = sorted(reforms, key=lambda x: x[0])
    j = 0

    for k in range(n_steps):
        tk = t[k]
        while j < len(reforms_sorted) and abs(reforms_sorted[j][0] - tk) < 0.5 * dt:
            _, dpsi = reforms_sorted[j]
            psi[k] = max(psi_L, psi[k] - float(dpsi))
            j += 1

        Wk = W_of_psi(psi[k]) - mu
        drift = float(sigma) * math.tanh(Wk / float(omega))
        psi[k + 1] = float(np.clip(psi[k] + dt * drift, psi_L, 1.0))

    return t, psi

def pick_psi0(psi_L, barpsi, psi0_buffer, psi_hi):
    psi_min = max(float(psi_L) + EPS_PSI_L, float(barpsi) + float(psi0_buffer))
    psi_cap = float(psi_hi) - float(EPS_PSI_HI)

    pinned = (float(barpsi) + float(psi0_buffer)) > psi_cap
    return float(min(psi_min, psi_cap)), pinned

print("Executing Candidate Audit Loop and bistable pool compilation...")
pool_rows = []
audit_rows = []
regime_counts = {drag_mode: {"bistable_interior": 0, "born_high": 0, "born_low": 0, "unclassified": 0, "unclassified_nonmonotone": 0, "unclassified_no_bistable_interval": 0}
                 for drag_mode in [True, False]}
structural_admissible_count = 0
threshold_location_feasible_count = 0

cfg_pool = dict(MC_CONFIG)
cfg_pool["psi_grid_n"] = max(PSI_GRID_N_LIST)

tries = 0
while tries < MAX_TRIES and len(pool_rows) < N_BISTABLE_POOL_TARGET:
    tries += 1
    theta = draw_admissible_primitives(rng, MC_CONFIG)
    dyn = draw_dyn_params(rng, MC_CONFIG)

    objs = build_manifold_objects(theta, cfg_pool, include_structural_drag=True)
    soc_global_margin_value = global_soc_margin(theta, MC_CONFIG)
    soc_global_ok = bool(np.isfinite(soc_global_margin_value) and soc_global_margin_value > SOC_TOL)

    solver_ok_endpoints = objs["solver_ok_endpoints"]
    excluded_bad_share = objs["excluded_bad_share"]
    omega_base = objs["omega_base"]

    omega_ok = np.isfinite(omega_base) and (omega_base > 0.0)
    admissible = bool(solver_ok_endpoints and (not excluded_bad_share) and omega_ok and objs["solv_ok"] and objs["gstar_ok"] and objs["mono_ok"] and objs["deriv_ok"] and soc_global_ok)

    barpsi, regime, W_floor, W_top = np.nan, "unclassified", np.nan, np.nan
    bistable_admissible = False
    threshold_location_feasible = False
    u_mu = np.nan
    mu_bistable_low = np.nan
    mu_bistable_low_raw = np.nan
    mu_bistable_high = np.nan
    mu_nonnegative_support_enforced = True

    if admissible:
        structural_admissible_count += 1
        valid_ref = np.asarray(objs["valid_mask"], dtype=bool)
        psi_ref = np.asarray(objs["psi_grid"], dtype=float)[valid_ref]
        W_ref = np.asarray(objs["W_star"], dtype=float)[valid_ref]
        if psi_ref.size >= 2 and psi_ref[0] <= theta["psi_L"] + 1e-12 and psi_ref[-1] >= 1.0 - 1e-12:
            W_floor = float(np.interp(theta["psi_L"], psi_ref, W_ref))
            W_top = float(np.interp(1.0, psi_ref, W_ref))
            mu_bistable_low_raw = W_floor + BISTABLE_MARGIN
            mu_bistable_low = max(0.0, mu_bistable_low_raw)
            mu_bistable_high = W_top - BISTABLE_MARGIN
            threshold_location_feasible = bool(np.isfinite(mu_bistable_low) and np.isfinite(mu_bistable_high) and (mu_bistable_high > mu_bistable_low))
        if threshold_location_feasible:
            threshold_location_feasible_count += 1
            u_mu = rng.uniform(*THRESHOLD_LOCATION_U_RANGE)
            dyn["mu_implied"] = float(mu_bistable_low + u_mu * (mu_bistable_high - mu_bistable_low))
            barpsi, regime, W_floor, W_top = find_barpsi(
                objs["psi_grid"], objs["W_star"], theta["psi_L"], dyn["mu_implied"], int(max(BARPSI_BISECT_ITERS_LIST)), objs["valid_mask"]
            )
        else:
            regime = "unclassified_no_bistable_interval"
        regime_counts[True][regime] += 1
        if regime == "bistable_interior":
            bistable_admissible = True

            row = {
                "candidate_idx": tries - 1,
                "theta": theta, "sigma": dyn["sigma"], "omega_mult": dyn["omega_mult"], "mu_implied": dyn["mu_implied"],
                "sample_design": SAMPLE_DESIGN, "u_mu": u_mu,
                "mu_bistable_low": mu_bistable_low, "mu_bistable_low_raw": mu_bistable_low_raw,
                "mu_bistable_high": mu_bistable_high,
                "mu_nonnegative_support_enforced": mu_nonnegative_support_enforced,
                "threshold_location_feasible": threshold_location_feasible,
                "soc_global_ok": soc_global_ok, "soc_global_margin": float(soc_global_margin_value),
                "regime_drag_1": regime, "barpsi_drag_1": barpsi, "W_floor_drag_1": W_floor, "W_top_drag_1": W_top,
                "W_minus_mu_floor_drag_1": W_floor - dyn["mu_implied"], "W_minus_mu_top_drag_1": W_top - dyn["mu_implied"]
            }

            objs_sens = build_manifold_objects(theta, cfg_pool, include_structural_drag=False)
            omega_base_sens = objs_sens["omega_base"]
            omega_ok_sens = np.isfinite(omega_base_sens) and (omega_base_sens > 0.0)
            admissible_sens = bool(objs_sens["solver_ok_endpoints"] and (not objs_sens["excluded_bad_share"]) and omega_ok_sens and objs_sens["solv_ok"] and objs_sens["gstar_ok"] and objs_sens["mono_ok"] and objs_sens["deriv_ok"] and soc_global_ok)
            if admissible_sens:
                barpsi_sens, regime_sens, W_floor_sens, W_top_sens = find_barpsi(
                    objs_sens["psi_grid"], objs_sens["W_star"], theta["psi_L"], dyn["mu_implied"], int(max(BARPSI_BISECT_ITERS_LIST)), objs_sens["valid_mask"]
                )
            else:
                barpsi_sens, regime_sens, W_floor_sens, W_top_sens = np.nan, "unclassified", np.nan, np.nan
            regime_counts[False][regime_sens] += 1

            row["admissible_drag_1"] = True
            row["admissible_drag_0"] = admissible_sens
            row["regime_drag_0"] = regime_sens
            row["barpsi_drag_0"] = barpsi_sens
            row["W_floor_drag_0"] = W_floor_sens
            row["W_top_drag_0"] = W_top_sens
            row["W_minus_mu_floor_drag_0"] = W_floor_sens - dyn["mu_implied"]
            row["W_minus_mu_top_drag_0"] = W_top_sens - dyn["mu_implied"]

            pool_rows.append(row)

    audit_rows.append({
        "candidate_idx": tries - 1,
        "structural_country_idx": len(pool_rows) - 1 if bistable_admissible else -1,
        "psi_L": theta["psi_L"], "Gamma": theta["Gamma"], "kappa_c": theta["kappa_c"],
        "rho_bar": theta["rho_bar"], "a": theta["a"], "b": theta["b"],
        "c_lambda": theta["c_lambda"], "eta_lambda": theta["eta_lambda"],
        "c_ell": theta["c_ell"], "eta_ell": theta["eta_ell"], "k": theta["k"], "s_N": theta["s_N"],
        "sigma": dyn["sigma"], "omega_mult": dyn["omega_mult"], "sample_design": SAMPLE_DESIGN,
        "mu_implied": dyn["mu_implied"], "u_mu": u_mu,
        "mu_bistable_low": mu_bistable_low, "mu_bistable_low_raw": mu_bistable_low_raw,
        "mu_bistable_high": mu_bistable_high,
        "mu_nonnegative_support_enforced": mu_nonnegative_support_enforced,
        "threshold_location_feasible": threshold_location_feasible,
        "omega": max(MC_CONFIG["OMEGA_MIN"], dyn["omega_mult"] * omega_base) if admissible else np.nan,
        "barpsi": barpsi,
        "W_floor": W_floor,
        "W_top": W_top,
        "W_minus_mu_floor": W_floor - dyn["mu_implied"] if np.isfinite(dyn["mu_implied"]) else np.nan,
        "W_minus_mu_top": W_top - dyn["mu_implied"] if np.isfinite(dyn["mu_implied"]) else np.nan,
        "threshold_class": regime,
        "solver_ok_endpoints": solver_ok_endpoints,
        "excluded_bad_share": excluded_bad_share,
        "bad_share": objs["bad_share"],
        "omega_ok": omega_ok,
        "solv_ok": objs["solv_ok"],
        "gstar_ok": objs["gstar_ok"],
        "mono_ok": objs["mono_ok"],
        "deriv_ok": objs["deriv_ok"],
        "soc_global_ok": soc_global_ok,
        "soc_global_margin": float(soc_global_margin_value),
        "admissible": admissible,
        "bistable_admissible": bistable_admissible,
        "bistable_margin": BISTABLE_MARGIN,
        "include_structural_drag_main": True,
    })

    if (tries % PROGRESS_EVERY_TRIES == 0) or (bistable_admissible and len(pool_rows) % PROGRESS_EVERY_ACCEPTED == 0):
        print(
            f"[pool] tries={tries:,} | accepted_bistable={len(pool_rows):,}/{N_BISTABLE_POOL_TARGET:,} | "
            f"structural_admissible={structural_admissible_count:,} | threshold_location_feasible={threshold_location_feasible_count:,} | "
            f"acceptance={len(pool_rows) / max(tries, 1):.3f}"
        )

df_audit = pd.DataFrame(audit_rows)
assert not df_audit.empty, "Audit ledger empty."

assert np.all(np.isfinite(df_audit.loc[df_audit["bistable_admissible"], "mu_implied"])), "Nonfinite implied threshold values among accepted cases."
save_df(df_audit, "candidate_audit")

flat_pool = []
for i, r in enumerate(pool_rows):
    th = r["theta"]
    out = {
        "candidate_idx": r["candidate_idx"],
        "structural_country_idx": i,
        "sigma": r["sigma"],
        "omega_mult": r["omega_mult"],
        "sample_design": r["sample_design"],
        "mu_implied": r["mu_implied"],
        "u_mu": r["u_mu"],
        "mu_bistable_low": r["mu_bistable_low"],
        "mu_bistable_low_raw": r["mu_bistable_low_raw"],
        "mu_bistable_high": r["mu_bistable_high"],
        "mu_nonnegative_support_enforced": r["mu_nonnegative_support_enforced"],
        "threshold_location_feasible": r["threshold_location_feasible"],
        "soc_global_ok": r["soc_global_ok"],
        "soc_global_margin": r["soc_global_margin"],
        **th
    }
    for drag_mode in [True, False]:
        out[f"admissible_drag_{int(drag_mode)}"] = r.get(f"admissible_drag_{int(drag_mode)}", False)
        out[f"regime_drag_{int(drag_mode)}"] = r[f"regime_drag_{int(drag_mode)}"]
        out[f"barpsi_drag_{int(drag_mode)}"] = r[f"barpsi_drag_{int(drag_mode)}"]
        out[f"W_floor_drag_{int(drag_mode)}"] = r[f"W_floor_drag_{int(drag_mode)}"]
        out[f"W_top_drag_{int(drag_mode)}"] = r[f"W_top_drag_{int(drag_mode)}"]
        out[f"W_minus_mu_floor_drag_{int(drag_mode)}"] = r[f"W_minus_mu_floor_drag_{int(drag_mode)}"]
        out[f"W_minus_mu_top_drag_{int(drag_mode)}"] = r[f"W_minus_mu_top_drag_{int(drag_mode)}"]
    flat_pool.append(out)

df_pool = pd.DataFrame(flat_pool)
assert len(df_pool) > 0, "No bistable countries generated."
pool_path = save_df(df_pool, "bistable_pool")

threshold_scale = pd.DataFrame({
    "barpsi_drag_1": df_pool["barpsi_drag_1"],
    "W_floor_drag_1": df_pool["W_floor_drag_1"],
    "W_top_drag_1": df_pool["W_top_drag_1"],
    "W_width_drag_1": df_pool["W_top_drag_1"] - df_pool["W_floor_drag_1"],
    "u_mu": df_pool["u_mu"],
    "mu_bistable_low_raw": df_pool["mu_bistable_low_raw"],
    "mu_bistable_low": df_pool["mu_bistable_low"],
    "mu_bistable_high": df_pool["mu_bistable_high"],
    "mu_implied": df_pool["mu_implied"],
})
summary_rows = []
for col in threshold_scale.columns:
    x = pd.to_numeric(threshold_scale[col], errors="coerce").dropna()
    summary_rows.append({
        "variable": col,
        "N": int(x.shape[0]),
        "mean": float(x.mean()),
        "sd": float(x.std(ddof=1)),
        "min": float(x.min()),
        "p10": float(x.quantile(0.10)),
        "p25": float(x.quantile(0.25)),
        "p50": float(x.quantile(0.50)),
        "p75": float(x.quantile(0.75)),
        "p90": float(x.quantile(0.90)),
        "max": float(x.max()),
    })
df_threshold_location_summary = pd.DataFrame(summary_rows)
save_df(df_threshold_location_summary, "threshold_location_summary")

reg_summ = []
for drag_mode in [True, False]:
    cnt = regime_counts[drag_mode]
    tot = sum(cnt.values())
    evaluation_domain = "structural_admissible_candidates" if drag_mode else "accepted_main_drag_pool"
    n_evaluated = int(structural_admissible_count if drag_mode else len(df_pool))
    reg_summ.append({
        "include_structural_drag": drag_mode,
        "evaluation_domain": evaluation_domain,
        "n_attempted": int(tries),
        "n_structural_admissible": int(structural_admissible_count),
        "n_threshold_location_feasible": int(threshold_location_feasible_count),
        "n_soc_global_rejected": int((~df_audit["soc_global_ok"]).sum()),
        "n_solvency_rejected": int((~df_audit["solv_ok"]).sum()),
        "minimum_soc_global_margin": float(df_audit["soc_global_margin"].min()),
        "n_evaluated": n_evaluated,
        "n_regime_counted": int(tot),
        **{f"count_{k}": v for k, v in cnt.items()},
        **{f"share_{k}_among_counted": (v / tot if tot > 0 else 0.0) for k, v in cnt.items()},
        **{f"share_{k}_among_attempted": (v / tries if tries > 0 else 0.0) for k, v in cnt.items()},
    })
df_reg = pd.DataFrame(reg_summ)
save_df(df_reg, "regime_summary")

print(f"Pool compiled: {len(df_pool)} admissible draws. Summary saved to: {OUT_DIR}")

_PROFILE_CACHE = {}

def profile_recheck_reason(objs):
    omega_ok = np.isfinite(objs["omega_base"]) and (objs["omega_base"] > 0.0)
    if not objs["solver_ok_endpoints"]:
        return "solver_endpoints"
    if objs["excluded_bad_share"]:
        return "bad_share"
    if not omega_ok:
        return "omega"
    if not objs["solv_ok"]:
        return "solvency"
    if not objs["gstar_ok"]:
        return "gstar"
    if not objs["mono_ok"]:
        return "monotonicity"
    if not objs["deriv_ok"]:
        return "derivative"
    return "ok"

def format_reason_counts(reason_counts):
    if not reason_counts:
        return ""
    return ";".join(f"{k}:{reason_counts[k]}" for k in sorted(reason_counts))

def get_scenario_profile(draw_row, psi_grid_n, drag_mode):
    key = (int(draw_row["candidate_idx"]), int(psi_grid_n), bool(drag_mode))
    if key in _PROFILE_CACHE:
        return _PROFILE_CACHE[key]

    theta = {k: float(draw_row[k]) for k in [
        "psi_L", "Gamma", "kappa_c", "rho_bar", "a", "b",
        "c_lambda", "eta_lambda", "c_ell", "eta_ell", "k", "s_N"
    ]}

    cfg_local = dict(MC_CONFIG)
    cfg_local["psi_grid_n"] = int(psi_grid_n)

    objs = build_manifold_objects(theta, cfg_local, include_structural_drag=drag_mode)
    recheck_reason = profile_recheck_reason(objs)
    profile_admissible = recheck_reason == "ok"
    if not profile_admissible:
        _PROFILE_CACHE[key] = (theta, objs["psi_grid"], None, None, None, None, objs["omega_base"], objs["W_star"], objs["valid_mask"], False, recheck_reason)
        return _PROFILE_CACHE[key]

    rK, Ge, gstar, W = make_grid_lookups(objs, theta)
    _PROFILE_CACHE[key] = (theta, objs["psi_grid"], rK, Ge, gstar, W, objs["omega_base"], objs["W_star"], objs["valid_mask"], True, recheck_reason)
    return _PROFILE_CACHE[key]

print("Executing Deliverable 1 Sweep...")
rows_d1 = []
reform_times = np.array(START_TIME_LIST, dtype=float)

for psi_grid_n in PSI_GRID_N_LIST:
    for bis_it in BARPSI_BISECT_ITERS_LIST:
        for include_drag in [True, False]:
            regime_col = f"regime_drag_{int(include_drag)}"
            admiss_col = f"admissible_drag_{int(include_drag)}"
            sub = df_pool[(df_pool[admiss_col]) & (df_pool[regime_col] == "bistable_interior")].copy()
            if sub.empty: continue

            for psi0_buffer in PSI0_BUFFER_LIST:
                for N_used in N_USED_LIST:
                    subN = sub.head(int(N_used))
                    deltas_by_t = {float(t): [] for t in reform_times}
                    pinned_count = 0
                    total_count = 0
                    skipped_profile_recheck = 0
                    skipped_barpsi = 0
                    profile_reason_counts = {}

                    for _, r in subN.iterrows():
                        theta, psi_grid, rK, Ge, gstar, W, omega_base, W_grid, valid_mask, profile_ok, recheck_reason = get_scenario_profile(r, psi_grid_n, include_drag)
                        if not profile_ok:
                            skipped_profile_recheck += 1
                            profile_reason_counts[recheck_reason] = profile_reason_counts.get(recheck_reason, 0) + 1
                            continue

                        barpsi, _, W_floor, W_top = find_barpsi(psi_grid, W_grid, theta["psi_L"], r["mu_implied"], int(bis_it), valid_mask)
                        if not np.isfinite(barpsi):
                            skipped_barpsi += 1
                            continue
                        assert np.isfinite(W_floor) and np.isfinite(W_top), "Any nonfinite barpsi or floor/top values in simulation loop."

                        psi0, pinned = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)
                        if pinned:
                            pinned_count += 1
                        total_count += 1

                        omega_val = float(max(MC_CONFIG["OMEGA_MIN"], r["omega_mult"] * omega_base))

                        _, psi_path = simulate_psi_path(
                            psi0, theta["psi_L"], W,
                            sigma=float(r["sigma"]), omega=omega_val, mu=float(r["mu_implied"]),
                            T=T, dt=DT, reforms=None
                        )

                        for tt in reform_times:
                            k_step = int(round(tt / DT))
                            k_step = max(0, min(k_step, len(psi_path) - 1))

                            deltas_by_t[float(tt)].append(max(0.0, float(psi_path[k_step] - barpsi)))

                    for tt in reform_times:
                        arr = np.asarray(deltas_by_t[float(tt)], dtype=float)

                        rows_d1.append({
                            "psi_grid_n": int(psi_grid_n),
                            "barpsi_bisect_iters": int(bis_it),
                            "include_structural_drag": bool(include_drag),
                            "sample_design": SAMPLE_DESIGN,
                            "estimand_scope": "conditional_bistable_threshold_location_domain",
                            "psi0_buffer": float(psi0_buffer),
                            "N_used": int(N_used),
                            "t": float(tt),
                            "hysteresis_margin_mean": float(np.mean(arr)) if arr.size > 0 else np.nan,
                            "hysteresis_margin_median": float(np.median(arr)) if arr.size > 0 else np.nan,
                            "hysteresis_margin_p25": float(np.percentile(arr, 25)) if arr.size > 0 else np.nan,
                            "hysteresis_margin_p75": float(np.percentile(arr, 75)) if arr.size > 0 else np.nan,
                            "N_requested": int(len(subN)),
                            "N_profile_valid": int(total_count + skipped_barpsi),
                            "N_skipped_profile_recheck": int(skipped_profile_recheck),
                            "profile_recheck_failure_reasons": format_reason_counts(profile_reason_counts),
                            "N_skipped_barpsi": int(skipped_barpsi),
                            "N": int(arr.size),
                            "share_pinned": float(pinned_count / total_count) if total_count > 0 else np.nan,
                        })

df_d1 = pd.DataFrame(rows_d1)
assert len(df_d1) > 0, "No D1 rows."
save_df(df_d1, "deliverable_1_delta_psi_crit")

print("Executing Deliverable 2 Sweep...")
rows_d2 = []

for psi_grid_n in PSI_GRID_N_LIST:
    for bis_it in BARPSI_BISECT_ITERS_LIST:
        for include_drag in [True, False]:
            regime_col = f"regime_drag_{int(include_drag)}"
            admiss_col = f"admissible_drag_{int(include_drag)}"
            sub = df_pool[(df_pool[admiss_col]) & (df_pool[regime_col] == "bistable_interior")].copy()
            if sub.empty: continue

            for psi0_buffer in PSI0_BUFFER_LIST:
                for dGamma in D_GAMMA_LIST:
                    for N_used in N_USED_LIST:
                        subN = sub.head(int(N_used))

                        dg_mod_list, dg_high_list = [], []
                        pinned_count = 0
                        total_count = 0
                        skipped_profile_recheck = 0
                        skipped_barpsi = 0
                        skipped_nonfinite_response = 0
                        profile_reason_counts = {}

                        for _, r in subN.iterrows():
                            theta, psi_grid, rK, Ge, gstar, W, _, W_grid, valid_mask, profile_ok, recheck_reason = get_scenario_profile(r, psi_grid_n, include_drag)
                            if not profile_ok:
                                skipped_profile_recheck += 1
                                profile_reason_counts[recheck_reason] = profile_reason_counts.get(recheck_reason, 0) + 1
                                continue

                            barpsi, _, _, _ = find_barpsi(psi_grid, W_grid, theta["psi_L"], r["mu_implied"], int(bis_it), valid_mask)
                            if not np.isfinite(barpsi):
                                skipped_barpsi += 1
                                continue

                            psi_mod, pinned = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)
                            psi_high = float(PSI_HI) - float(EPS_PSI_HI)

                            if pinned:
                                pinned_count += 1
                            total_count += 1

                            g_mod = gstar(psi_mod)
                            Ge_mod = Ge(psi_mod)
                            g_high = gstar(psi_high)
                            Ge_high = Ge(psi_high)

                            g_mod_stim = g_star_stationary(psi_mod, Ge_mod + float(dGamma), theta["kappa_c"])
                            g_high_stim = g_star_stationary(psi_high, Ge_high + float(dGamma), theta["kappa_c"])

                            dg_mod = float(g_mod_stim - g_mod)
                            dg_high = float(g_high_stim - g_high)

                            if np.isfinite(dg_mod) and np.isfinite(dg_high):
                                dg_mod_list.append(dg_mod)
                                dg_high_list.append(dg_high)
                            else:
                                skipped_nonfinite_response += 1

                        dg_mod_arr = np.asarray(dg_mod_list)
                        dg_high_arr = np.asarray(dg_high_list)
                        ratio_mask = dg_mod_arr > D2_RATIO_DENOM_MIN
                        ratio_arr = dg_high_arr[ratio_mask] / dg_mod_arr[ratio_mask]
                        ratio_median = float(np.median(ratio_arr)) if ratio_arr.size > 0 else np.nan

                        rows_d2.append({
                            "psi_grid_n": int(psi_grid_n),
                            "barpsi_bisect_iters": int(bis_it),
                            "include_structural_drag": bool(include_drag),
                            "sample_design": SAMPLE_DESIGN,
                            "estimand_scope": "conditional_bistable_threshold_location_domain",
                            "psi0_buffer": float(psi0_buffer),
                            "dGamma": float(dGamma),
                            "N_used": int(N_used),
                            "dg_mod_median": float(np.median(dg_mod_arr)) if dg_mod_arr.size > 0 else np.nan,
                            "dg_mod_p25": float(np.percentile(dg_mod_arr, 25)) if dg_mod_arr.size > 0 else np.nan,
                            "dg_mod_p75": float(np.percentile(dg_mod_arr, 75)) if dg_mod_arr.size > 0 else np.nan,
                            "dg_high_median": float(np.median(dg_high_arr)) if dg_high_arr.size > 0 else np.nan,
                            "dg_high_p25": float(np.percentile(dg_high_arr, 25)) if dg_high_arr.size > 0 else np.nan,
                            "dg_high_p75": float(np.percentile(dg_high_arr, 75)) if dg_high_arr.size > 0 else np.nan,
                            "attenuation_ratio_median": ratio_median,
                            "ratio_scope": "comparable_cases_dg_mod_gt_threshold",
                            "ratio_denom_min": float(D2_RATIO_DENOM_MIN),
                            "N_ratio_comparable": int(ratio_arr.size),
                            "N_ratio_excluded": int(len(dg_mod_arr) - ratio_arr.size),
                            "N_requested": int(len(subN)),
                            "N_profile_valid": int(total_count + skipped_barpsi),
                            "N_skipped_profile_recheck": int(skipped_profile_recheck),
                            "profile_recheck_failure_reasons": format_reason_counts(profile_reason_counts),
                            "N_skipped_barpsi": int(skipped_barpsi),
                            "N_skipped_nonfinite_response": int(skipped_nonfinite_response),
                            "N_eval": int(len(dg_mod_arr)),
                            "share_pinned": float(pinned_count / total_count) if total_count > 0 else np.nan,
                        })

df_d2 = pd.DataFrame(rows_d2)
assert len(df_d2) > 0, "No D2 rows."
save_df(df_d2, "deliverable_2_stimulus_attenuation")

print("Executing Deliverable 3 Sweep...")
rows_d3 = []

def run_d3_sweep_block(psi_grid_n, bis_it, psi0_buffer, include_drag, N_used_d3):
    regime_col = f"regime_drag_{int(include_drag)}"
    admiss_col = f"admissible_drag_{int(include_drag)}"
    sub = df_pool[(df_pool[admiss_col]) & (df_pool[regime_col] == "bistable_interior")].copy()
    if sub.empty: return

    subN = sub.head(int(N_used_d3))

    for freq in REFORM_FREQ_LIST:
        for max_ref in MAX_REFORMS_LIST:
            for dpsi in REFORM_SIZE_LIST:
                for t0 in START_TIME_LIST:
                    esc_terminal = 0
                    esc_confirmed = 0
                    tot = 0
                    pinned_count = 0
                    skipped_profile_recheck = 0
                    skipped_barpsi = 0
                    profile_reason_counts = {}
                    candidate_times = [float(t0 + k * freq) for k in range(int(max_ref))]
                    reform_times_active = [tt for tt in candidate_times if tt < T]
                    reforms = [(tt, float(dpsi)) for tt in reform_times_active]
                    n_ref = len(reforms)

                    for _, r in subN.iterrows():
                        theta, psi_grid, rK, Ge, gstar, W, omega_base, W_grid, valid_mask, profile_ok, recheck_reason = get_scenario_profile(r, psi_grid_n, include_drag)
                        if not profile_ok:
                            skipped_profile_recheck += 1
                            profile_reason_counts[recheck_reason] = profile_reason_counts.get(recheck_reason, 0) + 1
                            continue

                        barpsi, _, _, _ = find_barpsi(psi_grid, W_grid, theta["psi_L"], r["mu_implied"], int(bis_it), valid_mask)
                        if not np.isfinite(barpsi):
                            skipped_barpsi += 1
                            continue

                        psi0, pinned = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)
                        if pinned:
                            pinned_count += 1

                        omega_val = float(max(MC_CONFIG["OMEGA_MIN"], r["omega_mult"] * omega_base))

                        t_steps, psi_path = simulate_psi_path(
                            psi0, theta["psi_L"], W,
                            sigma=float(r["sigma"]), omega=omega_val, mu=float(r["mu_implied"]),
                            T=T, dt=DT, reforms=reforms
                        )

                        escaped_term = bool(psi_path[-1] <= barpsi - ESCAPE_TOL)

                        k_window = int(round(CONFIRM_WINDOW / DT))
                        tail_path = psi_path[-k_window:]
                        escaped_conf = bool(np.all(tail_path <= barpsi - ESCAPE_TOL))

                        esc_terminal += int(escaped_term)
                        esc_confirmed += int(escaped_conf)
                        tot += 1

                    if tot > 0:
                        P_term = esc_terminal / tot
                        P_conf = esc_confirmed / tot
                    else:
                        P_term = np.nan
                        P_conf = np.nan

                    if np.isfinite(P_term):
                        assert 0.0 <= P_term <= 1.0, "Any escape probability outside [0,1]."
                    if np.isfinite(P_conf):
                        assert 0.0 <= P_conf <= 1.0, "Any escape probability outside [0,1]."

                    rows_d3.append({
                        "psi_grid_n": int(psi_grid_n),
                        "barpsi_bisect_iters": int(bis_it),
                        "include_structural_drag": bool(include_drag),
                        "sample_design": SAMPLE_DESIGN,
                        "estimand_scope": "conditional_bistable_threshold_location_domain_grid_cell",
                        "probability_weighting": "unweighted_reform_grid_cell",
                        "psi0_buffer": float(psi0_buffer),
                        "N_used": int(N_used_d3),
                        "reform_size": float(dpsi),
                        "t_start": float(t0),
                        "reform_freq": float(freq),
                        "max_reforms": int(max_ref),
                        "n_reforms_active": int(n_ref),
                        "escaped_terminal": float(P_term),
                        "escaped_confirmed": float(P_conf),
                        "N_requested": int(len(subN)),
                        "N_profile_valid": int(tot + skipped_barpsi),
                        "N_skipped_profile_recheck": int(skipped_profile_recheck),
                        "profile_recheck_failure_reasons": format_reason_counts(profile_reason_counts),
                        "N_skipped_barpsi": int(skipped_barpsi),
                        "N_eval": int(tot),
                        "share_pinned": float(pinned_count / tot) if tot > 0 else np.nan,
                    })

for include_drag in [True, False]:
    for N_used_d3 in N_USED_D3_LIST:
        run_d3_sweep_block(
            psi_grid_n=401,
            bis_it=90,
            psi0_buffer=0.03,
            include_drag=include_drag,
            N_used_d3=N_used_d3
        )

df_d3 = pd.DataFrame(rows_d3)
assert len(df_d3) > 0, "No D3 rows."
save_df(df_d3, "deliverable_3_escape_grid")

print("Generating monochrome figures...")
plt.style.use('grayscale')

def spec_label(include_drag):
    return "Main specification: structural drag" if include_drag else "Sensitivity: closure only"

if not df_d1.empty:
    ref_d1 = df_d1[
        (df_d1["psi_grid_n"] == 401) &
        (df_d1["barpsi_bisect_iters"] == 90) &
        (df_d1["psi0_buffer"] == 0.03) &
        (df_d1["N_used"] == max(N_USED_LIST))
    ]
    if not ref_d1.empty:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        for i, drag in enumerate([True, False]):
            ax = axes[i]
            sub = ref_d1[ref_d1["include_structural_drag"] == drag]
            if not sub.empty:
                t = sub["t"].values
                med = sub["hysteresis_margin_median"].values
                p25 = sub["hysteresis_margin_p25"].values
                p75 = sub["hysteresis_margin_p75"].values

                ax.plot(t, med, 'o-', color="black", linewidth=2, label="Median")
                ax.fill_between(t, p25, p75, color="gray", alpha=0.25, label="Interquartile Range")
                ax.set_xlabel("Time (Years)")
                ax.set_ylabel("Hysteresis Margin (psi_t - barpsi)")
                ax.set_title(f"Hysteresis Margin - {spec_label(drag)}")
                ax.legend(frameon=True, facecolor="white")
                ax.grid(True, alpha=0.3)
        plt.tight_layout()
        save_fig(fig, "D1_hysteresis_margin_time")

if not df_d2.empty:
    ref_d2 = df_d2[
        (df_d2["psi_grid_n"] == 401) &
        (df_d2["barpsi_bisect_iters"] == 90) &
        (df_d2["psi0_buffer"] == 0.03) &
        (df_d2["N_used"] == max(N_USED_LIST))
    ]
    if not ref_d2.empty:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        for i, drag in enumerate([True, False]):
            ax = axes[i]
            sub = ref_d2[ref_d2["include_structural_drag"] == drag]
            if not sub.empty:
                dG = sub["dGamma"].values
                dg_mod = sub["dg_mod_median"].values
                dg_high = sub["dg_high_median"].values

                x = np.arange(len(dG))
                w = 0.35
                ax.bar(x - w/2, dg_mod, w, color="gray", label="Near-Threshold (psi_mod)")
                ax.bar(x + w/2, dg_high, w, color="black", label="Extreme (psi_high)")
                ax.set_xlabel("Stimulus Size (dGamma)")
                ax.set_ylabel("Growth Response (dg*)")
                ax.set_title(f"Growth Response Attenuation - {spec_label(drag)}")
                ax.set_xticks(x)
                ax.set_xticklabels([f"{val:.3f}" for val in dG])
                ax.legend(frameon=True, facecolor="white")
                ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        save_fig(fig, "D2_stimulus_response_attenuation")

if not df_d3.empty:
    ref_d3 = df_d3[
        (df_d3["psi_grid_n"] == 401) &
        (df_d3["barpsi_bisect_iters"] == 90) &
        (df_d3["psi0_buffer"] == 0.03) &
        (df_d3["N_used"] == max(N_USED_D3_LIST))
    ]
    if not ref_d3.empty:
        for drag in [True, False]:
            for freq in REFORM_FREQ_LIST:
                for max_ref in MAX_REFORMS_LIST:
                    sub = ref_d3[
                        (ref_d3["include_structural_drag"] == drag) &
                        (ref_d3["reform_freq"] == freq) &
                        (ref_d3["max_reforms"] == max_ref)
                    ]
                    if sub.empty: continue

                    pivot = sub.pivot_table(
                        values="escaped_confirmed",
                        index="reform_size",
                        columns="t_start",
                        aggfunc="first"
                    )
                    if pivot.empty: continue

                    fig, ax = plt.subplots(figsize=(8, 5.5))
                    im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="gray", vmin=0.0, vmax=1.0)

                    ax.set_xticks(np.arange(len(pivot.columns)))
                    ax.set_yticks(np.arange(len(pivot.index)))
                    ax.set_xticklabels([f"{v:.0f}" for v in pivot.columns])
                    ax.set_yticklabels([f"{v:.2f}" for v in pivot.index])

                    ax.set_xlabel("Reform Start Time")
                    ax.set_ylabel("Reform Size (dpsi)")
                    ax.set_title(f"Finite-Horizon Escape - {spec_label(drag)}, Frequency {int(freq)}, Cap {max_ref}")

                    cbar = plt.colorbar(im, ax=ax)
                    cbar.set_label("Confirmed Escape Rate (Grid Cell)")

                    plt.tight_layout()
                    save_fig(fig, f"D3_escape_heatmap_drag_{int(drag)}_freq_{int(freq)}_cap_{max_ref}")

def _sanitize(x):
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, (np.floating, np.integer)): return x.item()
    return x

meta_config = {
    "MC_NAME": MC_NAME,
    "RUN_TS": RUN_TS,
    "SEED": SEED,
    "MANUSCRIPT_SCOPE": {
        "main_results": "include_structural_drag=True",
        "sensitivity_results": "include_structural_drag=False",
        "sample_design": SAMPLE_DESIGN,
        "estimand_scope": "conditional on structurally admissible bistable economies with sampled threshold locations",
        "threshold_parameterization": "u_mu is drawn inside the bistable wedge interval; mu_implied is derived, not independently drawn",
        "mu_implied_rule": "mu_bistable_low = max(0, W_floor + BISTABLE_MARGIN); mu_bistable_high = W_top - BISTABLE_MARGIN; u_mu is drawn from THRESHOLD_LOCATION_U_RANGE; mu_implied = mu_bistable_low + u_mu * (mu_bistable_high - mu_bistable_low)",
        "mu_support": "nonnegative threshold support enforced before drawing u_mu",
        "d2_ratio_scope": "attenuation ratio computed only where dg_mod exceeds D2_RATIO_DENOM_MIN",
        "d3_probability_scope": "escape rates are unweighted reform-grid-cell rates, not unconditional policy probabilities",
    },
    "EXPLICIT_PARAMS": {
        "N_BISTABLE_POOL_TARGET": N_BISTABLE_POOL_TARGET,
        "MAX_TRIES": MAX_TRIES,
        "PROGRESS_EVERY_TRIES": PROGRESS_EVERY_TRIES,
        "PROGRESS_EVERY_ACCEPTED": PROGRESS_EVERY_ACCEPTED,
        "SOC_TOL": SOC_TOL,
        "SOC_RHO_MIN": SOC_RHO_MIN,
        "SOC_GRID_N": SOC_GRID_N,
        "SOC_REFINE_N": SOC_REFINE_N,
        "BASE_OUT_DIR": BASE_OUT_DIR,
        "T": T,
        "DT": DT,
        "BISTABLE_MARGIN": BISTABLE_MARGIN,
        "ESCAPE_TOL": ESCAPE_TOL,
        "CONFIRM_WINDOW": CONFIRM_WINDOW,
        "D2_RATIO_DENOM_MIN": D2_RATIO_DENOM_MIN,
        "PSI_GRID_N_LIST": PSI_GRID_N_LIST,
        "BARPSI_BISECT_ITERS_LIST": BARPSI_BISECT_ITERS_LIST,
        "PSI0_BUFFER_LIST": PSI0_BUFFER_LIST,
        "N_USED_LIST": N_USED_LIST,
        "N_USED_D3_LIST": N_USED_D3_LIST,
        "SAMPLE_DESIGN": SAMPLE_DESIGN,
        "THRESHOLD_LOCATION_U_RANGE": THRESHOLD_LOCATION_U_RANGE,
        "D_GAMMA_LIST": D_GAMMA_LIST,
        "REFORM_SIZE_LIST": REFORM_SIZE_LIST,
        "START_TIME_LIST": START_TIME_LIST,
        "REFORM_FREQ_LIST": REFORM_FREQ_LIST,
        "MAX_REFORMS_LIST": MAX_REFORMS_LIST,
    },
    "MC_CONFIG": {k: _sanitize(v) for k, v in MC_CONFIG.items()},
    "SIM_CONFIG": {k: _sanitize(v) for k, v in SIM_CONFIG.items()},
    "plots": PLOTS,
    "platform": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
}
with open(os.path.join(OUT_DIR, "config_dump.json"), "w", encoding="utf-8") as f:
    json.dump(meta_config, f, indent=2)

print("\n" + "=" * 100)
print("mc_block3_intervention_hysteresis - SIMULATION COMPLETE")
print("=" * 100)
print(f"Output files stored in: {OUT_DIR}")
print("=" * 100)
